# Домашнее задание 2
Краткий анализ датасета Titanic.

## Загрузка данных
Считываем таблицу из файла `train.csv`.

In [2]:
import re
import pandas as pd

df = pd.read_csv('train.csv')
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Основная информация
Смотрим типы данных, пропуски и основные статистики.

In [3]:
print(f'Размер датасета: {df.shape[0]} строк и {df.shape[1]} столбцов')
print()
df.info()
print()
print('Пропуски по столбцам:')
display(df.isna().sum())
print('Средние и другие статистики по числовым столбцам:')
display(df.describe())
print('Статистики по категориальным столбцам:')
display(df.describe(include='object'))


Размер датасета: 891 строк и 12 столбцов

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Пропуски по столбцам:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


Средние и другие статистики по числовым столбцам:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Статистики по категориальным столбцам:


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Dooley, Mr. Patrick",male,347082,G6,S
freq,1,577,7,4,644


## Выживаемость по классам
Считаем долю выживших в процентах для каждого `Pclass`.

In [4]:
survival_by_class = (df.groupby('Pclass')['Survived'].mean() * 100).round(2)
survival_by_class


,Survived
Pclass,
1,62.96
2,47.28
3,24.24


## Подготовка имен
Выделяем имя пассажира из поля `Name`.

In [5]:
def extract_first_name(name: str, sex: str) -> str:
    match = re.search(r'\(([^)]+)\)', name)
    if sex == 'female' and match:
        return match.group(1).strip().split()[0].strip('"\'.,')
    part = name.split(',', 1)[1].strip() if ',' in name else name
    if '. ' in part:
        part = part.split('. ', 1)[1]
    return part.split()[0].strip('"\'.,')

df['FirstName'] = df.apply(lambda row: extract_first_name(row['Name'], row['Sex']), axis=1)
df[['Name', 'Sex', 'FirstName']].head()


,Name,Sex,FirstName
0,"Braund, Mr. Owen Harris",male,Owen
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,Florence
2,"Heikkinen, Miss. Laina",female,Laina
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,Lily
4,"Allen, Mr. William Henry",male,William


## Самые популярные имена
Считаем отдельно для мужчин и женщин.

In [6]:
def top_names(data: pd.DataFrame) -> pd.Series:
    counts = data['FirstName'].value_counts()
    return counts[counts == counts.max()]

print('Самые популярные мужские имена:')
display(top_names(df[df['Sex'] == 'male']))
print('Самые популярные женские имена:')
display(top_names(df[df['Sex'] == 'female']))


Самые популярные мужские имена:


,count
FirstName,
William,35


Самые популярные женские имена:


,count
FirstName,
Mary,15
Anna,15


## Популярные имена по классам
Смотрим лидеров по имени для каждого пола внутри каждого класса.

In [7]:
rows = []

for pclass in sorted(df['Pclass'].dropna().unique()):
    for sex in ['male', 'female']:
        subset = df[(df['Pclass'] == pclass) & (df['Sex'] == sex)]
        counts = subset['FirstName'].value_counts()
        top = counts[counts == counts.max()]
        rows.append({
            'Pclass': pclass,
            'Sex': sex,
            'TopNames': ', '.join(top.index.tolist()),
            'Count': int(top.iloc[0])
        })

pd.DataFrame(rows)


,Pclass,Sex,TopNames,Count
0,1,male,William,11
1,1,female,"Elizabeth, Margaret",5
2,2,male,William,9
3,2,female,Elizabeth,5
4,3,male,William,15
5,3,female,Anna,9


## Пассажиры старше 44 лет
Выводим часть таблицы по условию.

In [8]:
df[df['Age'] > 44].head(10)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
54,55,0,1,"Ostby, Mr. Engelhart Cornelius",male,65.0,0,1,113509,61.9792,B30,C,Engelhart
62,63,0,1,"Harris, Mr. Henry Birkhardt",male,45.0,1,0,36973,83.4750,C83,S,Henry
92,93,0,1,"Chaffee, Mr. Herbert Fuller",male,46.0,1,0,W.E.P. 5734,61.1750,E31,S,Herbert
94,95,0,3,"Coxon, Mr. Daniel",male,59.0,0,0,364500,7.2500,NaN,S,Daniel
96,97,0,1,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C,George


## Мужчины младше 44 лет
Выводим часть таблицы по двум условиям.

In [9]:
df[(df['Age'] < 44) & (df['Sex'] == 'male')].head(10)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
16,17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.125,NaN,Q,Eugene
20,21,0,2,"Fynney, Mr. Joseph J",male,35.0,0,0,239865,26.000,NaN,S,Joseph
21,22,1,2,"Beesley, Mr. Lawrence",male,34.0,0,0,248698,13.000,D56,S,Lawrence
23,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.500,A6,S,William
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.000,C23 C25 C27,S,Charles


## Вместимость кабин
Считаем, сколько кают были 2-местными, 3-местными и далее.

In [10]:
cabin_counts = df['Cabin'].dropna().str.split().explode().value_counts()
multi_seat_cabins = cabin_counts.value_counts().sort_index()
multi_seat_cabins = multi_seat_cabins[multi_seat_cabins.index > 1]
multi_seat_cabins.rename_axis('PlacesInCabin').reset_index(name='CabinsCount')


,PlacesInCabin,CabinsCount
0,2,44
1,3,6
2,4,7


## Пассажиры без родственников
Считаем тех, у кого `SibSp + Parch = 0`

In [11]:
((df['SibSp'] + df['Parch']) == 0).sum()


np.int64(537)